# 7 задание

1. Загрузите объекты из новостного датасета 20 newsgroups, относящиеся к категориям "космос" и "атеизм".

In [22]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.svm import SVC

newsgroups = fetch_20newsgroups(subset='all',
                                categories=['alt.atheism', 'sci.space'],
                                download_if_missing=True,
                                shuffle=True,
                                random_state=42)
x_text = newsgroups.data
y = newsgroups.target

2. Вычислите TF-IDF-признаки для всех текстов.

In [23]:
vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(x_text)
print("Размер матрицы (документы, признаки):", x.shape)

Размер матрицы (документы, признаки): (1786, 28382)


3. Подберите минимальный лучший параметр C из множества [10^−5, 10^−4, ... 10^4, 10^5] для SVM с линейным ядром (kernel=’linear’) при помощи кросс-валидации по 5 блокам. Укажите параметр random_state=241 и для SVM, и для KFold. В качестве меры качества используйте долю верных ответов (accuracy).

In [24]:
C_values = [10**i for i in range(-5, 6)]
setka_param = {'C': C_values}


cv = KFold(n_splits=5, shuffle=True, random_state=241)
svm = SVC(kernel='linear', random_state=241)
gs = GridSearchCV(svm, setka_param, scoring='accuracy', cv=cv)

gs.fit(x, y)
print("Лучший параметр C:", gs.best_params_['C'])
print("Лучшая средняя accuracy:", gs.best_score_)

Лучший параметр C: 1
Лучшая средняя accuracy: 0.9932804406678872


In [28]:
best_score = gs.best_score_
best_C_candidates = [params['C'] for params, score in zip(gs.cv_results_['params'], gs.cv_results_['mean_test_score'])
                     if score == best_score]

best_c = min(best_C_candidates)
print(f"Максимальное качество: {best_score:.4f}")
print(f"Минимальный лучший C: {best_c}")

Максимальное качество: 0.9933
Минимальный лучший C: 1


4. Обучите SVM по всей выборке с лучшим параметром C, найденным на предыдущем шаге.

In [29]:
model = SVC(kernel='linear', C=best_c, random_state=241)
model.fit(x, y)

SVC(C=1, kernel='linear', random_state=241)

5. Найдите 10 слов с наибольшим по модулю весом. Они являются ответом на это задание. Укажите их через запятую, в нижнем регистре, в лексикографическом порядке.

In [30]:
coef = model.coef_[0]
feature_names = vectorizer.get_feature_names_out()
pairs = [(word, abs(weight)) for word, weight in zip(feature_names, coef)]

sorted_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)
words_10 = [word for word, weight in sorted_pairs[:10]]

In [31]:
sorted_10_words = sorted(words_10)

otvet = ','.join(sorted_10_words)

print("Количество пар:", len(pairs))
print("Первые 5 слов по весу:", [word for word, _ in sorted_pairs[:5]])
print("sorted_10_words:", sorted_10_words)

Количество пар: 1
Первые 5 слов по весу: ['00']
sorted_10_words: ['00']


сохранение ответа

In [33]:

with open('answer.txt', 'w') as f:
    f.write(otvet)

#8 задание


1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [ ]:
import pandas as pd
import pandas as pd
from sklearn.metrics import roc_auc_score
import numpy as np

data = pd.read_csv('/content/data-logistic.csv')
x = data.iloc[:, 1:3].values
y = data.iloc[:, 0].values

2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска.

3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

In [ ]:
k = 0.1
C = 0.0
max_iter = 10000
tol = 1e-5
def train_logistic_regression(x, y, C, k, max_iter, tol):
  w = np.array([0.0, 0.0])
  for i in range(max_iter):
    s = x.dot(w)
    z = -y * s
    exp = np.exp(z)

    t = 1.0 / (1.0 + exp)
    minus_t = 1 - t
    grad = np.mean((y * minus_t)[:, np.newaxis] * x, axis=0)
    grad_reg = grad - C * w
    w_new = w + k * grad_reg


4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.

In [17]:

    diff = np.linalg.norm(w_new - w)
    w = w_new
    if diff < tol:
        break






0.927 0.936


5.Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании?

In [ ]:
  s_final = x.dot(w)
  prob = 1.0 / (1.0 + np.exp(-s_final))
  y_bin = (y == 1).astype(int)
  auc = roc_auc_score(y_bin, prob)

  return w, auc

Сохранение ответа в файл

In [ ]:

answer = f"{round(auc_no_reg, 3)} {round(auc_reg, 3)}"
print("Ответ:", answer)

with open('answer.txt', 'w') as f:
    f.write(answer)